# 02b · GasWindTunnel: R2 handcrafted features

**Purpose.** Compute k = 12 features per sensor (R2). Fit the 4 heads on both tasks with the same folds and seeds as 01a.

**Inputs**
- GasWindTunnel `X_signals.npy` (600, 72, 1500) + `meta.csv`

**Outputs**
- F_hand: (600, 72, 12) → (600, 864)
- `results/02b_gwt_handcrafted.csv`

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.handcrafted import FEATURE_NAMES, handcrafted_features
DATASET = "gwt"
REP = "R2_handcrafted"
NAME = "02b_gwt_handcrafted"          # results/02b_gwt_handcrafted.csv

X, meta, info = load_processed(DATASET)    # X: (N, C, T) baseline corrected
print(X.shape, meta["y_multi"].value_counts().sort_index().tolist())

## R2: 12 handcrafted features per sensor
Nothing is fitted, so the features are computed once for all samples.

In [ ]:
levels = load_baseline_level(DATASET)     # (N, C) resting level before baseline correction
F = handcrafted_features(X, levels, info["n_base"], info["exposure_end"])   # (N, C * 12) = (600, 864)
assert F.shape == (600, 864) and np.isfinite(F).all()
pd.DataFrame(F[:, :12], columns=FEATURE_NAMES).describe().T   # the 12 features of the first sensor

## Cross validation: 4 heads × 5 folds × 3 seeds, both tasks
Everything that is fitted (scaler, heads, hyperparameters) sees the training fold only.

In [ ]:
scores, preds = [], []
for task in TASKS:                                   # "multi", then "binary"
    s, p = run_cv(F, meta, DATASET, task, REP, precomputed)
    scores.append(s)
    preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)
print("saved results/" + NAME + ".csv")

## Results: mean ± std over 3 seeds × 5 folds

In [ ]:
print("macro F1"); display(summarize(scores))
print("balanced accuracy"); display(summarize(scores, "balanced_acc"))

In [ ]:
print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")